# 🏛️ **Expansive Tokenization & Contextual Embedding Workbook**
### *Tokenization Built Upon Context of Interest — From First Text Generation to Custom Persona, Dialect, and Fandom Tokenization Sets*

[![Repository: evecount/expansive_tokenization](https://img.shields.io/badge/GitHub-evecount%2Fexpansive__tokenization-181717?logo=github)](https://github.com/evecount/expansive_tokenization)
[![Framework: Hugging Face Transformers](https://img.shields.io/badge/%F0%9F%A4%97%20Transformers-Tokenizers%20%26%20768D%20Embeddings-ffb000)](https://huggingface.co/docs/transformers)

---

## 💡 **Why This Workbook Exists: Tokenization Is Classification by Context of Interest**

Standard language models use a **fixed, universal subword vocabulary** (such as Byte-Pair Encoding or WordPiece). While convenient, a static tokenizer treats every conversation as if the speaker and listener share the exact same default worldview.

In reality:
1. **Tokenization is an act of ontological framing:** If you asked a human to *"tokenize a turkey dinner"*, they wouldn't split it into arbitrary character chunks (`["tur", "##key"]`). They would decompose the scene into meaningful functional slots: **`roast_turkey`**, **`mashed_potatoes`**, **`roasted_pumpkin`**, **`pan_gravy`**, **`cranberry_sauce`**, **`convection_oven`**, and **`warmed_ceramic_plates`**.
2. **Subword Fragmentation shatters human passions & dialects:** When a general tokenizer encounters **Greek Mythology & Epic Poetry** (`katabasis`, `kleos`, `wine-dark_sea`), **Egyptology** (`ma'at`, `ushabti`), **Edgar Allan Poe's Gothic lexicon** (`tintinnabulation`, `amontillado`), **LARPing & Fandom lore** (`phys-rep`, `bleed_effect`, `headcanon`), or regional dialects (**Québécois French**, **Singlish**, **PT-PT vs. PT-BR**, **Catalan**, **Northern English**), it shatters those core concepts into meaningless fragments.
3. **Interlocutor-Aware Token Embeddings (768-D Vector Shifts):** Even when two people use the **exact same English or French word**—like *`attention`*, *`weights`*, *`cooked`*, *`bet`*, *`boot`*, *`char`*, or *`canny`*—its 768-dimensional embedding vector shifts dramatically depending on whether you are speaking to a **General Speaker vs. a Machine Learning Researcher**, a **Millennial vs. Gen-Z**, or a speaker in **Québec vs. Paris**.

---

### 🗺️ **Workbook Roadmap**
- **Part 1 — Foundations: Generating First Text (`Phi-3-mini-4k-instruct`)**
- **Part 2 — Inside the Tokenizer: Subword IDs & The Turkey Dinner Scene Decomposition**
- **Part 3 — 768-Dimensional Token Embeddings (`bert-base-uncased`) & Contextual Vector Geometry**
- **Part 4 — The 8 Context-of-Interest Tokenization Packs (Subword Fertility & Fracture Audit)**
  1. 🏛️ *Greek Mythology & Epic Poetry* (`katabasis`, `kleos`, `nostos`, `xenia`, `rhapsode`)
  2. 🏺 *Egyptology, Edgar Allan Poe Gothic & LARP / Fandoms* (`ma'at`, `tintinnabulation`, `phys-rep`, `bleed_effect`)
  3. 🧬 *Interlocutor Shift: General Speaker vs. ML Researcher* (`attention_head`, `gradient_descent`, `lora_adapter`)
  4. ⚡ *Generational Slang: Millennial vs. Gen-Z* (`adulting`, `on_fleek` vs. `no_cap`, `rizz`, `delulu`, `cooked`)
  5. 🌍 *World Englishes & Commonwealth Nuances* (`shiok`, `kiasu`, `chuffed`, `arvo`, `toque`)
  6. ⚜️ *Francophone Diatopic Nuances: Québec vs. Paris vs. Francophonie* (`char`, `dépanneur`, `tiguidou`, `chelou`, `teranga`)
  7. 🧭 *Iberian, Lusophone & Germanic Regional Continuums* (`autocarro`/`ônibus`, `seny`/`rauxa`, `canny`, `bairn`, `scran`)
  8. 🦃 *Sensory & Culinary Scene Decomposition ('The Turkey Dinner Experiment')*
- **Part 5 — Live Tokenizer Expansion & Compositional Semantic Embedding Initialization**
- **Part 6 — Measuring 768-D Contextual Embedding Shifts Across Personas, Generations & Dialects**
- **Part 7 — Interactive Tokenization Set Generator: Build & Export Your Own Context Pack!**


---
# **Part 1: Generating First Text (Causal Language Modeling)**

Before we customize how a language model carves up the world, let's see how a modern instruction-tuned Causal Language Model (`microsoft/Phi-3-mini-4k-instruct`) loads its default tokenizer and generates text.


In [1]:
# Import the required classes from the Hugging Face Transformers library
# AutoModelForCausalLM: Used for loading a pre-trained causal (text-generation) model
# AutoTokenizer: Used for converting text into tokens (numbers) that the model can understand
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

# Load the model and matching tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Loaded {model_name} on device: {model.device} | Base vocab size: {len(tokenizer):,}")


Loaded microsoft/Phi-3-mini-4k-instruct on device: cuda:0 | Base vocab size: 32,011


In [2]:
# Create a deterministic text-generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=200,
    do_sample=False,
)

messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
output = generator(messages, use_cache=False)
print(output[0]["generated_text"])


 Why did the chicken join the band? Because it had the drumsticks!


---
# **Part 2: Inside Tokenization & 'The Turkey Dinner Experiment'**

How does a tokenizer actually prepare inputs for a language model? It maps raw strings into integer `input_ids`. 

Let's first inspect a standard sentence—and then compare it to **The Turkey Dinner Experiment**:
> *"If I say: let me tokenize a turkey dinner—I would tokenize it based on the appearance of turkey, potatoes, pumpkins, gravy, cranberries, and then oven and plates."*

Let's see how a standard tokenizer chops words into morphological fragments (`apolog` + `izing`, `garden` + `ing`) vs. how we can define an **Ontological Scene Tokenization Set**.


In [3]:
# 1. Standard subword tokenization inspection
prompt = "Write a short email apologizing to Sarah for a small gardening accident."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

print("Raw input_ids tensor:", input_ids)
print("\nDecoded piece-by-piece:")
decoded_pieces = [tokenizer.decode(id_val) for id_val in input_ids[0]]
print(" | ".join(decoded_pieces))


Raw input_ids tensor: tensor([[14350,   263,  3273,  4876, 27746,  5281,   304, 19235,   363,   263,
          2319, 16423,   292, 11423, 29889]], device='cuda:0')

Decoded piece-by-piece:
Write | a | short | email | apolog | izing | to | Sarah | for | a | small | garden | ing | accident | .


In [4]:
# 2. The Turkey Dinner Experiment: Character Subwords vs. Ontological Scene Tokenization
turkey_scene = (
    "Out of the convection_oven came the roast_turkey with a crackling maillard_crust, "
    "served on warmed_ceramic_plates with mashed_potatoes, roasted_pumpkin, pan_gravy, and cranberry_sauce."
)

turkey_ontology = {
    "<|PROTEIN_CENTERPIECE|>": ["roast_turkey", "maillard_crust"],
    "<|STARCH_AND_HARVEST|>":  ["mashed_potatoes", "roasted_pumpkin", "herb_stuffing"],
    "<|SAUCE_AND_ACID|>":      ["pan_gravy", "cranberry_sauce"],
    "<|VESSEL_AND_HEAT|>":     ["convection_oven", "roasting_pan", "warmed_ceramic_plates", "gravy_boat"],
}

print("=== TURKEY DINNER ONTOLOGICAL SLOTS ===")
for slot, items in turkey_ontology.items():
    print(f"{slot:26s} -> {items}")


=== TURKEY DINNER ONTOLOGICAL SLOTS ===
<|PROTEIN_CENTERPIECE|>    -> ['roast_turkey', 'maillard_crust']
<|STARCH_AND_HARVEST|>     -> ['mashed_potatoes', 'roasted_pumpkin', 'herb_stuffing']
<|SAUCE_AND_ACID|>         -> ['pan_gravy', 'cranberry_sauce']
<|VESSEL_AND_HEAT|>        -> ['convection_oven', 'roasting_pan', 'warmed_ceramic_plates', 'gravy_boat']


---
# **Part 3: 768-Dimensional Token Embeddings (`bert-base-uncased`)**

Tokenization is only the first step. Once text is segmented into tokens, each token is projected into a **high-dimensional vector space** (for BERT-base, **768 dimensions**).

**Crucial Insight:** In modern Transformers, a token embedding is *not* a static dictionary lookup. It is dynamically computed from **how each word in the sentence relates to every other word**. Notice below how the two occurrences of the word **`the`** in `"The cat sat on the mat"` produce **different 768-dimensional vectors** because of their different grammatical positions!


In [5]:
from transformers import AutoTokenizer, AutoModel
import torch

bert_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name)
bert_model.eval()

text = "The cat sat on the mat"
inputs = bert_tokenizer(text, return_tensors="pt")
tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)

with torch.no_grad():
    outputs = bert_model(**inputs)

embeddings = outputs.last_hidden_state
print("Shape of embeddings (batch_size, sequence_length, embedding_dim):", embeddings.shape)
print("\nFirst 5 dimensions of each token's 768-D contextual vector:")
for tok, emb in zip(tokens, embeddings[0]):
    vals = ", ".join(f"{v:+.4f}" for v in emb[:5].tolist())
    print(f"  {tok:10s} -> [{vals}, ...]")


Tokens: ['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '[SEP]']
Shape of embeddings (batch_size, sequence_length, embedding_dim): torch.Size([1, 8, 768])

First 5 dimensions of each token's 768-D contextual vector:
  [CLS]      -> [-0.2027, +0.1071, -0.0085, +0.1114, -0.1550, ...]
  the        -> [-0.3376, -0.2804, -0.1846, +0.0244, +0.7472, ...]
  cat        -> [-0.3106, -0.0439, +0.2795, -0.1275, +0.5369, ...]
  sat        -> [+0.0585, -0.1808, +0.3539, +0.1578, +0.2722, ...]
  on         -> [-0.9252, -0.6113, +0.4229, +0.5525, +0.6041, ...]
  the        -> [-0.5056, -0.2047, -0.1147, +0.5620, +0.5953, ...]
  mat        -> [-0.6513, -0.4152, -0.3943, +0.5900, +0.2449, ...]
  [SEP]      -> [+0.7182, +0.1087, -0.3343, +0.5742, -0.4940, ...]


---
# **Part 4: Auditing the 8 Context-of-Interest Tokenization Sets**

What happens when a general-purpose tokenizer (`bert-base-uncased`) encounters specialized vocabulary from:
1. 🏛️ **Greek Mythology & Epic Poetry** (`katabasis`, `kleos`, `nostos`, `xenia`, `aristeia`, `rhapsode`)
2. 🏺 **Egyptology, Edgar Allan Poe Gothic & LARP / Fandoms** (`ma'at`, `ushabti`, `tintinnabulation`, `amontillado`, `phys-rep`, `bleed_effect`)
3. 🧬 **Machine Learning vs. Everyday Speech** (`attention_head`, `gradient_descent`, `lora_adapter`, `subword_fertility`)
4. ⚡ **Generational Slang: Millennial vs. Gen-Z** (`adulting`, `on_fleek`, `no_cap`, `rizz`, `delulu`, `ate_and_left_no_crumbs`)
5. 🌍 **World Englishes & Commonwealth Nuances** (`shiok`, `kiasu`, `kopitiam`, `chuffed`, `knackered`, `arvo`, `toque`)
6. ⚜️ **Francophone Diatopic Nuances (Québec vs. Paris vs. Francophonie)** (`dépanneur`, `tiguidou`, `poudrerie`, `chelou`, `teranga`)
7. 🧭 **Iberian, Lusophone & Germanic Continuums (PT-PT vs. PT-BR, Spanish/Catalan, Regional UK)** (`autocarro`/`ônibus`, `sobremesa`, `seny`/`rauxa`, `canny`, `bairn`, `scran`)
8. 🦃 **Sensory & Culinary Scene Decomposition ('The Turkey Dinner Experiment')**

Let's run a **Subword Fertility & Semantic Fracture Audit** across all 8 Context Packs using our `expansive_tokenization` engine!


In [6]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))

from expansive_tokenization import (
    CONTEXT_PACKS,
    list_context_packs,
    get_context_pack,
    audit_tokenization_fragmentation,
)

print(f"{'Context Pack Title':58s} | {'Terms':5s} | {'Fertility':9s} | {'Fracture %':10s}")
print("-" * 92)
for pack_id, pack in CONTEXT_PACKS.items():
    report = audit_tokenization_fragmentation(bert_tokenizer, pack)
    print(
        f"{pack.title[:58]:58s} | {report['num_domain_terms']:5d} | "
        f"{report['baseline_subword_fertility']:7.2f}x  | {report['semantic_fracture_rate_pct']:8.1f}%"
    )


Context Pack Title                                         | Terms | Fertility | Fracture %
--------------------------------------------------------------------------------------------
Greek Mythology & Epic Poetry                              |    25 |    3.12x  |     96.0%
Interlocutor Adaptation: General Speaker vs. ML Researcher |    21 |    3.67x  |    100.0%
Generational Slang: Millennial vs. Gen-Z Semantic Drift    |    27 |    2.63x  |     66.7%
World Englishes & Commonwealth Lexical Sets                |    26 |    2.54x  |     84.6%
Francophone Diatopic Nuances: Québec vs. Paris vs. Francop |    23 |    2.78x  |     95.7%
Sensory & Culinary Scene Decomposition ('The Turkey Dinner |    16 |    3.69x  |    100.0%
Iberian, Lusophone & Germanic Regional Continuums (PT-PT v |    26 |    2.88x  |     88.5%
Hyper-Niche Interest Sets: Egyptology, Edgar Allan Poe Got |    27 |    3.22x  |     96.3%


In [7]:
# Inspect exactly how standard tokenization shatters Greek Mythology, Poe Gothic, LARP & Québécois terms:
sample_inspect_terms = [
    ("Greek Myth", "katabasis"),
    ("Greek Myth", "dactylic_hexameter"),
    ("Greek Myth", "rosy-fingered_dawn"),
    ("Egyptology", "ushabti"),
    ("Poe Gothic", "tintinnabulation"),
    ("Poe Gothic", "amontillado"),
    ("LARP/Fandom", "bleed_effect"),
    ("Gen-Z Slang", "ate_and_left_no_crumbs"),
    ("Singlish", "kopitiam"),
    ("Québécois", "dépanneur"),
    ("Québécois", "tiguidou"),
    ("Lusophone", "pequeno-almoço"),
    ("Germanic/UK", "Fingerspitzengefühl"),
]

print(f"{'Domain':13s} | {'Atomic Concept':24s} | {'Fractured Subwords in Standard Tokenizer'}")
print("-" * 88)
for domain, term in sample_inspect_terms:
    pieces = bert_tokenizer.tokenize(term)
    print(f"{domain:13s} | {term:24s} | {pieces} ({len(pieces)} tokens)")


Domain        | Atomic Concept           | Fractured Subwords in Standard Tokenizer
----------------------------------------------------------------------------------------
Greek Myth    | katabasis                | ['kata', '##bas', '##is'] (3 tokens)
Greek Myth    | dactylic_hexameter       | ['da', '##ct', '##yl', '##ic', '_', 'hex', '##ame', '##ter'] (8 tokens)
Greek Myth    | rosy-fingered_dawn       | ['rosy', '-', 'fingered', '_', 'dawn'] (5 tokens)
Egyptology    | ushabti                  | ['us', '##hab', '##ti'] (3 tokens)
Poe Gothic    | tintinnabulation         | ['tin', '##tin', '##na', '##bul', '##ation'] (5 tokens)
Poe Gothic    | amontillado              | ['am', '##ont', '##illa', '##do'] (4 tokens)
LARP/Fandom   | bleed_effect             | ['bleed', '_', 'effect'] (3 tokens)
Gen-Z Slang   | ate_and_left_no_crumbs   | ['ate', '_', 'and', '_', 'left', '_', 'no', '_', 'crumb', '##s'] (10 tokens)
Singlish      | kopitiam                 | ['ko', '##pit', '##iam'] (3 toke

---
# **Part 5: Expanding a Live Tokenizer & Compositional Semantic Embedding Initialization**

Simply calling `tokenizer.add_tokens(...)` is not enough—if you don't initialize the new rows in the model's embedding matrix, the model assigns **random noise vectors** to your new tokens!

Our function `expand_tokenizer_with_context_set(...)` solves this with **Compositional Semantic Averaging**:
1. It adds the pack's **ontological category markers** (e.g., `<|ETHOS_AND_FATE|>`, `<|COSMOLOGY_AND_RITE|>`) as special control tokens.
2. It adds all **domain concepts** (`katabasis`, `kleos`, `nostos`, `wine-dark_sea`) as single atomic tokens.
3. It resizes the Transformer's embedding matrix and initializes each new token's 768-D vector to the **mean embedding of its semantic definition**, placing it immediately in the right geometric neighborhood!


In [8]:
from expansive_tokenization import expand_tokenizer_with_context_set
import copy

greek_pack = get_context_pack("greek_mythology_poetry")
model_copy = copy.deepcopy(bert_model)

expanded_tok, expanded_model, expansion_summary = expand_tokenizer_with_context_set(
    tokenizer=bert_tokenizer,
    model=model_copy,
    token_set=greek_pack,
    initialize_by_semantic_average=True,
)

print(f"Original Vocabulary Size : {expansion_summary['original_vocab_size']:,}")
print(f"Expanded Vocabulary Size : {expansion_summary['new_vocab_size']:,} (+{expansion_summary['domain_atomic_tokens_added']} atomic domain tokens)")
print(f"Sample Corpus Tokens     : {expansion_summary['corpus_tokens_before']} -> {expansion_summary['corpus_tokens_after']} ({expansion_summary['token_savings_pct']}% reduction!)")

sample_verse = greek_pack.sample_corpus[0]
print("\nSample Verse:", sample_verse)
print("BEFORE Expansion:", bert_tokenizer.tokenize(sample_verse))
print("AFTER Expansion :", expanded_tok.tokenize(sample_verse))


Original Vocabulary Size : 30,522
Expanded Vocabulary Size : 30,550 (+25 atomic domain tokens)
Sample Corpus Tokens     : 109 -> 68 (37.6% reduction!)

Sample Verse: The rhapsode chanted in dactylic_hexameter of Odysseus yearning for nostos across the wine-dark_sea.
BEFORE Expansion: ['the', 'rh', '##aps', '##ode', 'chanted', 'in', 'da', '##ct', '##yl', '##ic', '_', 'hex', '##ame', '##ter', 'of', 'od', '##ys', '##seus', 'yearning', 'for', 'nos', '##tos', 'across', 'the', 'wine', '-', 'dark', '_', 'sea', '.']
AFTER Expansion : ['the', 'rhapsode', 'chanted', 'in', 'dactylic_hexameter', 'of', 'od', '##ys', '##seus', 'yearning', 'for', 'nostos', 'across', 'the', 'wine-dark_sea', '.']


---
# **Part 6: Measuring 768-D Contextual Embedding Shifts Across Personas, Generations & Dialects**

Now let's test the second half of our hypothesis:
> *"Even when the exact same surface word is used, its 768-dimensional embedding vector shifts depending on the model's understanding of the person and context it is engaging with."*

Below, we pass identical target words (`attention`, `weights`, `cooked`, `bet`, `boot`, `scheme`, `char`, `canny`, `bleed`, `shade`) in two contrasting contexts through BERT's 12 self-attention layers and measure their **Cosine Similarity** and **Semantic Shift Distance ($1 - \cos\theta$)** in 768-D space.


In [9]:
from expansive_tokenization import compute_contextual_embedding_shift

all_contrasts = []
for pack in CONTEXT_PACKS.values():
    all_contrasts.extend(pack.contrast_pairs)

shift_results = compute_contextual_embedding_shift(bert_tokenizer, bert_model, all_contrasts)

print(f"{'Target Word':12s} | {'Context A vs. Context B':54s} | {'Cosine Sim':10s} | {'Shift Dist':10s}")
print("-" * 95)
for res in shift_results:
    pair_label = f"{res['context_a_label'][:25]} vs. {res['context_b_label'][:25]}"
    print(
        f"{res['target_word']:12s} | {pair_label:54s} | "
        f"{res['cosine_similarity']:10.4f} | {res['semantic_shift_distance']:10.4f}"
    )


Target Word  | Context A vs. Context B                                | Cosine Sim | Shift Dist
-----------------------------------------------------------------------------------------------
shade        | Everyday Modern English vs. Homeric Underworld / Kata  |     0.5418 |     0.4582
weaver       | Industrial Textile Trade vs. Mythic Fate & Guile (Pen  |     0.6712 |     0.3288
attention    | General Everyday Speaker vs. Machine Learning Researc  |     0.5843 |     0.4157
weights      | General Everyday Speaker vs. Machine Learning Researc  |     0.5194 |     0.4806
loss         | General Everyday Speaker vs. Machine Learning Researc  |     0.5629 |     0.4371
cooked       | Literal Culinary / Millen vs. Gen-Z Slang (Doomed vs.  |     0.5310 |     0.4690
bet          | Traditional Gambling / Wa vs. Gen-Z Affirmative Disco  |     0.4892 |     0.5108
ate          | Literal Consumption vs. Gen-Z Aesthetic Praise         |     0.5764 |     0.4236
boot         | North American English (F

---
# **Part 7: The Custom Tokenization Set Generator (Build Your Own Pack!)**

Use the template cell below to design a custom **Tokenization Set** for *any* community, fandom, literary oeuvre, academic field, or regional dialect, test its compression & embedding initialization, and export it to JSON.


In [10]:
from expansive_tokenization import TokenizationSet, ContrastPair, export_tokenization_set_json

# 🛠️ Customize your own Context-of-Interest Tokenization Set here:
my_custom_set = TokenizationSet(
    pack_id="custom_fandom_or_dialect",
    title="My Custom Fandom / Dialect Tokenization Set",
    persona_or_domain="Custom Worldbuilder & Linguist",
    description="A bespoke tokenization set tailored to my community's vocabulary and semantic nuances.",
    ontological_categories={
        "<|CORE_CONCEPTS|>": ["world_seed", "lore_anchor", "harmonic_resonance"],
        "<|COMMUNITY_IDIOMS|>": ["canon_lock", "in_universe", "deep_lore"],
    },
    semantic_definitions={
        "world_seed": "foundational mythic premise from which a fictional universe grows",
        "lore_anchor": "immutable historical event that grounds character backstory",
        "harmonic_resonance": "synchronized alignment of thematic motifs across chapters",
    },
    sample_corpus=[
        "Every great campaign begins with a world_seed and an unshakeable lore_anchor in_universe.",
    ],
    contrast_pairs=[
        ContrastPair(
            target_word="anchor",
            context_a_label="Maritime Sailing",
            sentence_a="The captain dropped the heavy iron anchor into the harbor bay.",
            context_b_label="Worldbuilding Narrative Lore",
            sentence_b="That ancient prophecy serves as the primary anchor for the trilogy's timeline.",
            semantic_note="Shifts from physical maritime hardware to narrative structural foundation.",
        )
    ],
)

audit = audit_tokenization_fragmentation(bert_tokenizer, my_custom_set)
print(f"Custom Pack '{my_custom_set.title}' audited!")
print(f"Baseline Fertility: {audit['baseline_subword_fertility']}x | Fracture Rate: {audit['semantic_fracture_rate_pct']}%")

# Export to portable JSON
export_path = export_tokenization_set_json(my_custom_set, "custom_tokenization_set.json")
print(f"Exported portable specification to: {export_path}")


Custom Pack 'My Custom Fandom / Dialect Tokenization Set' audited!
Baseline Fertility: 3.17x | Fracture Rate: 100.0%
Exported portable specification to: custom_tokenization_set.json
